# 8.2 Coding Practice: Recommendation Systems and Personalized Text Ranking (Solution)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module8/week08_recommendation_practice_solution.ipynb)

This notebook continues the reading's running example. **Alex** knows Python and SQL and wants an early-career reporting role. All job descriptions, ratings, and relevance judgments below are teaching examples, not real postings or measured outcomes.

You will build the same job collection into three recommendation approaches:

1. **Content-based filtering** with TF-IDF and cosine similarity (Parts 1-2)
2. **Collaborative filtering** with a small user-job ratings table (Part 3)
3. **BM25 ranking**, compared against the TF-IDF/cosine baseline (Part 4)

Part 5 asks you to compare the three approaches and apply them to a new scenario.


## Learning goals

- Explain the difference between content-based and collaborative filtering.
- Represent text descriptions using TF-IDF and interpret the resulting feature matrix.
- Calculate and interpret cosine similarity for a content-based recommendation.
- Compare users with sparse rating data and produce one collaborative-filtering recommendation.
- Apply BM25 to rank job descriptions against a search query, using the `rank-bm25` library.
- Compare BM25 and TF-IDF/cosine rankings on the same query and explain why they differ.
- Compare recommendation methods and recognize their limitations.

## Setup

Run the next cell once. It installs `rank-bm25`, the only package Colab does not already include, using the same guarded pattern as the Week 06 notebook (check first, install quietly if missing). Everything else in this notebook uses `pandas`, `numpy`, and `scikit-learn`, which Colab already provides.

In [1]:
import importlib.util
import subprocess
import sys

# One-time setup. rank-bm25 is small and pure Python; this is the only
# install in the notebook. Safe to rerun.
if importlib.util.find_spec("rank_bm25") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank-bm25"])


In [2]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42


## 1. Content-based filtering: inspect the job collection

The job collection below extends the six jobs from the reading (`J1`-`J6`) with six new ones (`J7`-`J12`), still fit for Alex to search. Some are strong matches for Alex's early-career Python/SQL reporting goal; some look similar on the surface but require senior or management experience the reading already warned about; two are unrelated maintenance jobs included as a sanity check (an unrelated job should score near zero).

In [3]:
jobs = pd.DataFrame([
    {"job_id": "J1", "title": "Junior Data Analyst",
     "description": "Early-career data analyst role. Use Python and SQL to build recurring reports for the operations team. New graduates welcome; on-the-job training provided."},
    {"job_id": "J2", "title": "Analytics Director",
     "description": "Lead the analytics strategy using Python and SQL. Requires ten or more years of experience managing a data team and setting department priorities."},
    {"job_id": "J3", "title": "Research Data Assistant",
     "description": "Support a research group with Python and SQL reporting on language data. NLP training will be provided; no prior NLP experience required."},
    {"job_id": "J4", "title": "SQL Reporting Analyst",
     "description": "Build SQL database reports and apply basic text tagging to customer feedback. Early-career role with a supportive onboarding program."},
    {"job_id": "J5", "title": "Senior NLP Engineer",
     "description": "Build and deploy Python language models in production. Requires several years of hands-on experience shipping NLP systems at scale."},
    {"job_id": "J6", "title": "Equipment Maintenance Technician",
     "description": "Inspect and maintain manufacturing machinery. Mechanical aptitude required; no programming or data experience needed."},
    {"job_id": "J7", "title": "Junior SQL Analyst",
     "description": "Entry-level analyst writing SQL queries and building dashboards. No management responsibilities; ideal for a first data role."},
    {"job_id": "J8", "title": "Data Science Intern",
     "description": "Learn Python, Python, and more Python while supporting the data science team with small reporting tasks. Early-career, mentorship-focused internship."},
    {"job_id": "J9", "title": "Senior Project Manager",
     "description": "Manage cross-functional technical projects. Some familiarity with Python is helpful, but the role is centered on people management and long-term planning."},
    {"job_id": "J10", "title": "NLP Research Scientist",
     "description": "Conduct advanced NLP research with large language models. PhD or equivalent research experience required; focus on publishing novel methods."},
    {"job_id": "J11", "title": "Business Intelligence Reporting Analyst",
     "description": "Maintain SQL-based reporting dashboards for business stakeholders. Early-career friendly, with limited opportunity to grow into modeling work."},
    {"job_id": "J12", "title": "Warehouse Equipment Technician",
     "description": "Perform routine maintenance on warehouse equipment. Hands-on mechanical work; no data or text skills involved."},
]).set_index("job_id")

jobs


,title,description
job_id,,
J1,Junior Data Analyst,Early-career data analyst role. Use Python and SQL to build recurring reports for the operations team. New graduates...
J2,Analytics Director,Lead the analytics strategy using Python and SQL. Requires ten or more years of experience managing a data team and ...
J3,Research Data Assistant,Support a research group with Python and SQL reporting on language data. NLP training will be provided; no prior NLP...
J4,SQL Reporting Analyst,Build SQL database reports and apply basic text tagging to customer feedback. Early-career role with a supportive on...
J5,Senior NLP Engineer,Build and deploy Python language models in production. Requires several years of hands-on experience shipping NLP sy...
J6,Equipment Maintenance Technician,Inspect and maintain manufacturing machinery. Mechanical aptitude required; no programming or data experience needed.
J7,Junior SQL Analyst,Entry-level analyst writing SQL queries and building dashboards. No management responsibilities; ideal for a first d...
J8,Data Science Intern,"Learn Python, Python, and more Python while supporting the data science team with small reporting tasks. Early-caree..."
J9,Senior Project Manager,"Manage cross-functional technical projects. Some familiarity with Python is helpful, but the role is centered on peo..."


Alex's profile is one short piece of text, written the same way a job description is written, so it can be compared using the same vocabulary and weighting.

In [4]:
alex_profile = (
    "Early-career candidate skilled in Python and SQL, interested in reporting roles. "
    "Open to learning NLP but does not want a senior management position."
)
alex_profile


'Early-career candidate skilled in Python and SQL, interested in reporting roles. Open to learning NLP but does not want a senior management position.'

### TODO 1: Fit TF-IDF on the job collection

Fit a `TfidfVectorizer` on the `description` column of `jobs`, then transform Alex's profile into the *same* vector space (do not re-fit on the profile).

In [5]:
tfidf = TfidfVectorizer(stop_words="english")
job_tfidf = tfidf.fit_transform(jobs["description"])
alex_tfidf = tfidf.transform([alex_profile])

print("Job feature matrix shape:", job_tfidf.shape)
print("Alex profile vector shape:", alex_tfidf.shape)


Job feature matrix shape: (12, 117)
Alex profile vector shape: (1, 117)


**Guided question:** The feature matrix has one row per job and one column per vocabulary term. Why must Alex's profile be *transformed* using the vectorizer already fit on the jobs, rather than fitting a brand-new vectorizer on the profile text alone?

**Suggested answer:** Cosine similarity only makes sense when both vectors live in the same feature space (same columns, same weighting). Fitting a separate vectorizer on the one-sentence profile would create a different, much smaller vocabulary with different IDF weights, so the two vectors would not be comparable. Transforming the profile through the vectorizer already fit on the jobs keeps both on the same axes.

## 2. Cosine similarity and a recommendation function

Now compare Alex's profile vector against every job vector and rank the jobs.

### TODO 2: Compute cosine similarity and rank the jobs

In [6]:
alex_scores = cosine_similarity(alex_tfidf, job_tfidf).flatten()

scored_jobs = jobs.copy()
scored_jobs["cosine_score"] = alex_scores
scored_jobs = scored_jobs.sort_values("cosine_score", ascending=False)
scored_jobs[["title", "cosine_score"]]


,title,cosine_score
job_id,,
J3,Research Data Assistant,0.391911
J8,Data Science Intern,0.363823
J11,Business Intelligence Reporting Analyst,0.282511
J1,Junior Data Analyst,0.229865
J4,SQL Reporting Analyst,0.187631
J7,Junior SQL Analyst,0.177334
J9,Senior Project Manager,0.154500
J5,Senior NLP Engineer,0.149281
J2,Analytics Director,0.101627


### TODO 3: Build a recommendation function

Write a function that returns the top-`k` jobs for a given profile vector, excluding nothing special here (there is no "query item" to exclude, since Alex's profile is not itself a job).

In [7]:
def recommend_jobs(profile_vector, job_matrix, jobs_df, k=5):
    scores = cosine_similarity(profile_vector, job_matrix).flatten()
    ranked = jobs_df.copy()
    ranked["cosine_score"] = scores
    return ranked.sort_values("cosine_score", ascending=False).head(k)[["title", "cosine_score"]]


recommend_jobs(alex_tfidf, job_tfidf, jobs, k=5)


,title,cosine_score
job_id,,
J3,Research Data Assistant,0.391911
J8,Data Science Intern,0.363823
J11,Business Intelligence Reporting Analyst,0.282511
J1,Junior Data Analyst,0.229865
J4,SQL Reporting Analyst,0.187631


**Guided questions:**

- Why might two jobs have high cosine similarity even though one is clearly a poor match for Alex (hint: look at `J2` and `J9`)?
- Does high text similarity guarantee a good recommendation?
- What information does TF-IDF miss that a human reader would notice immediately in the full description?

**Suggested answers:**

- `J2` and `J9` both share vocabulary with Alex's profile ("Python", "SQL", or general data-team language) while requiring senior/management experience the profile explicitly does not want. Shared words do not encode the seniority constraint.
- No. High similarity means shared, weighted vocabulary, not fit on constraints like required years of experience, management scope, or career level, none of which TF-IDF represents unless that language happens to appear in the text.
- TF-IDF has no notion of *requirements* versus *nice-to-haves*, no notion of contradicted constraints ("ten or more years" directly conflicts with "early-career"), and cannot tell that a word appearing once as a brief mention ("some familiarity with Python") carries different weight than a role built around that skill.

## 3. Collaborative filtering

Now set the job *text* aside. A small group of users has rated a subset of these jobs (1-5 scale). A blank cell means the user has not rated that job, **not** that they dislike it.

In [8]:
rating_job_ids = ["J1", "J2", "J3", "J4", "J5", "J7", "J9", "J10", "J11"]

ratings = pd.DataFrame(
    {
        "J1":  [5, 5, np.nan, np.nan, 3],
        "J2":  [np.nan, np.nan, np.nan, 5, 2],
        "J3":  [4, 4, 5, np.nan, np.nan],
        "J4":  [4, 5, np.nan, np.nan, 3],
        "J5":  [np.nan, np.nan, 4, 3, np.nan],
        "J7":  [5, 4, np.nan, np.nan, 3],
        "J9":  [np.nan, np.nan, 2, 5, np.nan],
        "J10": [np.nan, np.nan, 5, np.nan, np.nan],
        "J11": [np.nan, 5, np.nan, np.nan, np.nan],
    },
    index=["Alex", "Blair", "Casey", "Dana", "Evan"],
)[rating_job_ids]

ratings


,J1,J2,J3,J4,J5,J7,J9,J10,J11
Alex,5.0,NaN,4.0,4.0,NaN,5.0,NaN,NaN,NaN
Blair,5.0,NaN,4.0,5.0,NaN,4.0,NaN,NaN,5.0
Casey,NaN,NaN,5.0,NaN,4.0,NaN,2.0,5.0,NaN
Dana,NaN,5.0,NaN,NaN,3.0,NaN,5.0,NaN,NaN
Evan,3.0,2.0,NaN,3.0,NaN,3.0,NaN,NaN,NaN


### Predict before computing

Skim the table. Which user do you expect to be most similar to Alex, and why? Which jobs do you expect to be hard to compare because too few users rated them?

### TODO 4: Compare users with cosine similarity

Missing ratings are filled with 0 only so the similarity calculation can run. This is a simplification, not a claim that a missing rating means the lowest possible score. Compute Alex's cosine similarity to every other user.

In [9]:
ratings_filled = ratings.fillna(0)

alex_vector = ratings_filled.loc[["Alex"]]
other_users = ratings_filled.drop(index="Alex")

similarity_scores = cosine_similarity(alex_vector, other_users).flatten()
user_similarity = pd.Series(similarity_scores, index=other_users.index).sort_values(ascending=False)
user_similarity


Blair    0.864741
Evan     0.833032
Casey    0.263982
Dana     0.000000
dtype: float64

### TODO 5: Make one collaborative-filtering recommendation

Using Alex's most similar user, recommend a job Alex has **not** yet rated that the neighbor rated highly.

In [10]:
most_similar_user = user_similarity.index[0]
print("Most similar user to Alex:", most_similar_user, f"(cosine = {user_similarity.iloc[0]:.3f})")

alex_rated = ratings.loc["Alex"].dropna().index
neighbor_ratings = ratings.loc[most_similar_user].dropna()
candidate_jobs = neighbor_ratings.drop(index=alex_rated, errors="ignore")

if len(candidate_jobs) > 0:
    recommended_job_id = candidate_jobs.sort_values(ascending=False).index[0]
    print(f"Recommend {recommended_job_id} ({jobs.loc[recommended_job_id, 'title']}): "
          f"{most_similar_user} rated it {neighbor_ratings[recommended_job_id]:.0f}/5, "
          "and Alex has not rated it.")
else:
    print("No unseen job to recommend from this neighbor; their rated jobs overlap entirely with Alex's.")


Most similar user to Alex: Blair (cosine = 0.865)
Recommend J11 (Business Intelligence Reporting Analyst): Blair rated it 5/5, and Alex has not rated it.


**Guided questions:**

- How is this different from the content-based approach in Parts 1-2? What evidence does it use that TF-IDF never sees?
- What happens when a new user has no ratings at all? Could this method make a recommendation for them?
- The blank cells were filled with 0 to run the calculation. What could go wrong if 0 is later read as "this user dislikes the job," instead of "we don't know"?

**Suggested answers:**

- Collaborative filtering never looks at the job text at all; it only uses patterns in *who rated what*. Two users can be judged similar even if the jobs they rated use completely different wording, as long as their rating patterns line up.
- No, not with this method alone. A brand-new user with zero ratings has an all-zero row, which has no meaningful cosine similarity to anyone (the calculation returns 0 for every comparison). This is the cold-start problem; content-based filtering (Parts 1-2) can still work for that user if they provide profile text instead of ratings.
- Treating a filled-in 0 as a real negative rating would make every unrated job look like a disliked job, which could suppress jobs a user simply never got to rate, and would silently misrepresent "unobserved" as "rejected." The fill is only a trick to make the arithmetic run, not a modeling choice to rely on.

## 4. BM25 ranking and a direct comparison with TF-IDF/cosine

BM25 ranks documents against a search *query*, rather than a profile paragraph. Use the same 12 jobs and compare BM25's ranking to a TF-IDF/cosine ranking of the exact same query.

In [11]:
query = "early career Python SQL reporting role"


def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())


tokenized_jobs = [tokenize(desc) for desc in jobs["description"]]
tokenized_query = tokenize(query)
tokenized_jobs[:2]


[['early',
  'career',
  'data',
  'analyst',
  'role',
  'use',
  'python',
  'and',
  'sql',
  'to',
  'build',
  'recurring',
  'reports',
  'for',
  'the',
  'operations',
  'team',
  'new',
  'graduates',
  'welcome',
  'on',
  'the',
  'job',
  'training',
  'provided'],
 ['lead',
  'the',
  'analytics',
  'strategy',
  'using',
  'python',
  'and',
  'sql',
  'requires',
  'ten',
  'or',
  'more',
  'years',
  'of',
  'experience',
  'managing',
  'a',
  'data',
  'team',
  'and',
  'setting',
  'department',
  'priorities']]

### TODO 6: Compute BM25 scores

Use `BM25Okapi` from `rank-bm25` on the tokenized job collection, then score the tokenized query against it. You are not implementing the BM25 formula yourself; the library handles term frequency, inverse document frequency, and document-length normalization. Elasticsearch's documented defaults are `k1 = 1.2` and `b = 0.75`; `rank-bm25` uses the same defaults unless you pass different values.

In [12]:
bm25 = BM25Okapi(tokenized_jobs)
bm25_scores = bm25.get_scores(tokenized_query)

bm25_ranked = jobs.copy()
bm25_ranked["bm25_score"] = bm25_scores
bm25_ranked = bm25_ranked.sort_values("bm25_score", ascending=False)
bm25_ranked[["title", "bm25_score"]]


,title,bm25_score
job_id,,
J11,Business Intelligence Reporting Analyst,2.322769
J8,Data Science Intern,2.220544
J4,SQL Reporting Analyst,1.907966
J1,Junior Data Analyst,1.715026
J3,Research Data Assistant,0.955530
J7,Junior SQL Analyst,0.665957
J9,Senior Project Manager,0.595774
J2,Analytics Director,0.000000
J5,Senior NLP Engineer,0.000000


### TODO 7: Rank the same query with TF-IDF/cosine, side by side

Transform `query` through the **same** `tfidf` vectorizer from Part 1 (do not re-fit), score it against `job_tfidf`, and place the two rankings next to each other for comparison.

In [13]:
query_tfidf = tfidf.transform([query])
tfidf_query_scores = cosine_similarity(query_tfidf, job_tfidf).flatten()

comparison = jobs.copy()
comparison["bm25_score"] = bm25_scores
comparison["tfidf_cosine_score"] = tfidf_query_scores

bm25_top5 = comparison.sort_values("bm25_score", ascending=False).head(5)[["title", "bm25_score"]]
cosine_top5 = comparison.sort_values("tfidf_cosine_score", ascending=False).head(5)[["title", "tfidf_cosine_score"]]

print("Top 5 by BM25:")
display(bm25_top5)
print("\nTop 5 by TF-IDF/cosine:")
display(cosine_top5)


Top 5 by BM25:


,title,bm25_score
job_id,,
J11,Business Intelligence Reporting Analyst,2.322769
J8,Data Science Intern,2.220544
J4,SQL Reporting Analyst,1.907966
J1,Junior Data Analyst,1.715026
J3,Research Data Assistant,0.955530



Top 5 by TF-IDF/cosine:


,title,tfidf_cosine_score
job_id,,
J8,Data Science Intern,0.419016
J1,Junior Data Analyst,0.343270
J11,Business Intelligence Reporting Analyst,0.325368
J4,SQL Reporting Analyst,0.296563
J3,Research Data Assistant,0.231428


**Guided questions:**

- Where do the two rankings agree? Where do they put jobs in a different order?
- `J8` repeats the word "Python" three times in a short description. Look at how `J8` scores under BM25 versus TF-IDF/cosine. Does repeating a term help the same way under both methods?
- How does document length affect ranking? Compare a short job description to a longer one that mentions the same query terms about the same number of times.
- What happens if the query uses a synonym that is absent from a job description word-for-word (for example, "reporting" instead of a description that only says "dashboards")? Would either method catch that match?

**Suggested answers:**

- Both methods tend to agree that the clearly irrelevant jobs (`J6`, `J12`) score at or near the bottom, and that strong lexical matches like `J1`, `J4`, `J7` score well under both. They can disagree on the exact order among the moderately-matching jobs, because the two methods weight term frequency and document length differently.
- Not identically. BM25's term-frequency component saturates (`k1` controls how quickly extra repeats stop adding score), while TF-IDF's raw term frequency grows roughly linearly before the vector is normalized. Repeating "Python" three times helps `J8` under both, but the *size* of that boost is governed by different mechanics.
- BM25 explicitly normalizes by document length through the `b` parameter, directly rewarding a short, concentrated match over a long document repeating the same term. TF-IDF/cosine normalizes each document vector to unit length, which also discounts long documents, but through vector-length normalization rather than BM25's explicit length-adjustment term inside the score formula.
- Neither. Both BM25 and TF-IDF/cosine here are purely lexical: they only match the exact tokens present in the text. A description that only says "dashboards" produces a zero contribution for the query term "reporting," regardless of the method. Catching that connection requires a different representation (semantic/embedding-based matching), which the reading describes but which neither method here implements.

## 5. Reflection and comparison

Fill in the table below.

| Method | Input Data | Strength | Limitation |
|---|---|---|---|
| Content-based (TF-IDF + cosine) | Job text and a profile written the same way | ? | ? |
| Collaborative filtering | User-job ratings (no text at all) | ? | ? |
| BM25 | A search query and the job text collection | ? | ? |

**Suggested completion:**

| Method | Input Data | Strength | Limitation |
|---|---|---|---|
| Content-based (TF-IDF + cosine) | Job text and a profile written the same way | Works immediately for a brand-new user with no history, as long as they can describe what they want | Only matches on vocabulary already present in the text; misses unstated constraints and synonyms |
| Collaborative filtering | User-job ratings (no text at all) | Can surface jobs a text-based method would never connect, based purely on shared taste patterns | Fails for new users or new jobs with no ratings yet (cold start); needs enough overlapping ratings to be meaningful |
| BM25 | A search query and the job text collection | Rewards concentrated, length-normalized term matches; good for short, specific queries | Still purely lexical; cannot match a synonym or paraphrase that never appears in the text |

**Scenario questions:**

1. A brand-new user signs up with no ratings yet, but writes two sentences about what kind of role they want. Which method(s) from this notebook could make a recommendation for them right now, and which could not?
2. Alex later says: "Show me jobs most like J11, not jobs that match my profile." Which part of this notebook already computes something close to that request, and what would you need to change to answer it directly?

*Conceptual extension (not implemented here):* nearest-neighbor search (KNN) uses the same similarity idea as Part 2 and Part 3 to retrieve the closest items or users at scale, rather than scoring every item one at a time. Everything in this notebook is a small enough collection that a full comparison already works, which is itself why a dedicated KNN step was not needed.

**Suggested answers:**

1. Content-based filtering (Parts 1-2) can use the two sentences immediately, by transforming them through the already-fitted TF-IDF vectorizer, the same way Alex's profile was handled. Collaborative filtering (Part 3) cannot help this user yet; they have no ratings row, so every similarity score against them would be 0. BM25 (Part 4) could also work if the two sentences are treated as a query, for the same reason as TF-IDF.
2. The cosine similarity step in Part 2 is already *almost* this: instead of comparing Alex's profile vector to every job, you would compare `J11`'s own TF-IDF row to every other job's row, using the same `job_tfidf` matrix. This is item-to-item content-based filtering rather than profile-to-item.

## Validation checks

These checks confirm the notebook ran top to bottom with no hidden state.

In [14]:
assert len(jobs) == 12, "Expected 12 jobs in the collection."
assert job_tfidf.shape[0] == 12
assert len(recommend_jobs(alex_tfidf, job_tfidf, jobs, k=5)) == 5
assert ratings.shape == (5, 9), "Expected 5 users and 9 rated jobs."
assert most_similar_user == "Blair", "Expected Blair to be Alex's most similar neighbor in this dataset."
assert len(bm25_scores) == 12
assert len(tfidf_query_scores) == 12
print("All checks passed.")


All checks passed.
